In [17]:
import numpy as np
import scanpy as sc

adata = sc.read("../data/processed/adata_slice_100.h5ad")
patches = np.load("../data/processed/patches.npy")

print(patches.shape)
print(adata.shape)

(3332, 224, 224, 3)
(3332, 3213)


In [18]:
sc.pp.normalize_total(adata, target_sum=1e4)

C:\Users\dhanu\AppData\Local\Temp\ipykernel_28740\3661531845.py:1: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(adata, target_sum=1e4)


In [19]:
sc.pp.log1p(adata)

In [20]:
import scanpy as sc

sc.pp.highly_variable_genes(adata, n_top_genes=1000)

adata = adata[:, adata.var["highly_variable"]]
print(adata.shape)

C:\Users\dhanu\AppData\Local\Temp\ipykernel_28740\1726919609.py:3: UserWarning: `n_top_genes` (=1000) > number of normalized dispersions (=315), returning all genes with normalized dispersions.
  sc.pp.highly_variable_genes(adata, n_top_genes=1000)


(3332, 315)


In [21]:
patches = patches / 255.0

In [22]:
genes = adata.X.toarray()  # do this ONCE

In [23]:
import torch
from torch.utils.data import Dataset

class SpatialDataset(Dataset):
    def __init__(self, patches, genes):
        self.patches = patches
        self.genes = genes

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        img = self.patches[idx]
        gene = self.genes[idx]

        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        
        # Optional: ImageNet normalization for pretrained ResNet18
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = (img - mean) / std

        gene = torch.tensor(gene, dtype=torch.float32)

        return img, gene

In [24]:
from torch.utils.data import DataLoader

subset = 500

dataset = SpatialDataset(patches[:subset], genes[:subset])

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0 
)

In [25]:
import torch.nn as nn
import torchvision.models as models

class GenePredictor(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        
        resnet = models.resnet18(pretrained=True)
        
        # remove final layer
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        
        self.mlp = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        x = self.encoder(x)           # (B, 512, 1, 1)
        x = x.view(x.size(0), -1)    # (B, 512)
        x = self.mlp(x)
        return x

In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model = GenePredictor(output_dim=adata.shape[1]).to(device)
for param in model.encoder.parameters():
    param.requires_grad = True

cuda


c:\Users\dhanu\Documents\preliminary-gsoc\histMOE\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\dhanu\Documents\preliminary-gsoc\histMOE\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [27]:
import torch.optim as optim

mse = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [28]:
import torch

def pearson_corr(x, y):
    vx = x - x.mean(dim=1, keepdim=True)
    vy = y - y.mean(dim=1, keepdim=True)
    
    corr = (vx * vy).sum(dim=1) / (
        torch.sqrt((vx**2).sum(dim=1)) * torch.sqrt((vy**2).sum(dim=1)) + 1e-8
    )
    
    return corr.mean()

In [29]:
from tqdm import tqdm

epochs = 20

for epoch in range(epochs):
    total_loss = 0
    
    loop = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for imgs, genes in loop:
        imgs = imgs.to(device)
        genes = genes.to(device)

        preds = model(imgs)
        
        mse_loss = mse(preds, genes)
        corr_loss = 1 - pearson_corr(preds, genes)
        loss = mse_loss + 0.1 * corr_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 🔥 update progress bar
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} Avg Loss: {avg_loss:.4f}")

Epoch 1/20: 100%|██████████| 16/16 [00:03<00:00,  4.33it/s, loss=0.132]


Epoch 1 Avg Loss: 0.1500


Epoch 2/20: 100%|██████████| 16/16 [00:02<00:00,  6.86it/s, loss=0.128]


Epoch 2 Avg Loss: 0.1248


Epoch 3/20: 100%|██████████| 16/16 [00:02<00:00,  6.83it/s, loss=0.126]


...

Epoch 18 Avg Loss: 0.1108


Epoch 19/20: 100%|██████████| 16/16 [00:02<00:00,  7.52it/s, loss=0.124]


Epoch 19 Avg Loss: 0.1104


Epoch 20/20: 100%|██████████| 16/16 [00:02<00:00,  7.45it/s, loss=0.121]

Epoch 20 Avg Loss: 0.1096


In [30]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3060 Laptop GPU
